# Análise Facial de Perfis do YouTube

Este notebook realiza análise demográfica dos usuários e criadores de canais através de suas fotos de perfil do YouTube.

**Pipeline**: Download de fotos de perfil → Detecção e alinhamento facial (DeepFace) → Análise de idade e gênero (InsightFace)

**Resultado**: CSVs com idade e gênero de usuários e canais para análise demográfica da audiência e criadores

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import re
from concurrent.futures import ThreadPoolExecutor
import requests
from functools import partial
from deepface import DeepFace
import random
import cv2
import tensorflow as tf
from insightface.app import FaceAnalysis

# local onde as imagens serão salvas
users_image_path = './profile_files/profile_pictures'
channels_image_path = './profile_files/channels_pictures'

# log para o download das imagens
log_users = './profile_files/users_log.txt'
log_channels = './profile_files/channels_log.txt'

# local onde as imagens que contem rosto serão salvas
users_faces_path = './profile_files/profile_final'
channels_faces_path = './profile_files/channels_final'

# log para o download das imagens que contem rostos
log_faces_users = './profile_files/log_faces_users.txt'
log_faces_channels = './profile_files/log_faces_channels.txt'

# arquivo onde será salvo as informações de gênero e idade
csv_users_path = './profile_files/user_atrIBUTES_unico.csv'
csv_channels_path = './profile_files/channels_atrIBUTES_unico.csv'

2025-12-16 15:06:19.133391: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---
## 1. Download de Fotos de Perfil

Baixa fotos de perfil de usuários (comentadores) e canais (criadores) do YouTube:
- **URL Resize**: Aumenta resolução para 400x400 pixels para melhor qualidade de detecção
- **Paralelização**: ThreadPoolExecutor com 20-30 workers para download simultâneo
- **Sistema de Log**: Evita redownload de imagens já processadas
- **Dados**: Extrai URLs de profile_picture_url e author_profile_image_url dos datasets

In [ ]:
df_channels = pd.read_csv('cleaned_data/filtered_channels.csv')
df_comments = pd.read_csv('modelo/prediction/final_comments_match_cleaned_pred.csv')

In [ ]:
channels_image_url = df_channels[['channel_id', 'profile_picture_url']].drop_duplicates(subset='channel_id').rename(columns={'channel_id': 'id', 'profile_picture_url': 'url'})
users_image_url = df_comments[['author_channel_id', 'author_profile_image_url']].drop_duplicates(subset='author_channel_id').rename(columns={'author_channel_id': 'id', 'author_profile_image_url': 'url'})

In [ ]:
len(channels_image_url), len(users_image_url)

In [ ]:
def resize_url(url, new_size=400):
    if pd.isna(url):
        return url
    return re.sub(r'([=\-/]s)\d+', rf'\g<1>{new_size}', url)

channels_image_url['url'] = channels_image_url['url'].apply(resize_url)
users_image_url['url'] = users_image_url['url'].apply(resize_url)

In [ ]:
len(channels_image_url), len(users_image_url)

In [ ]:
def load_log(arquivo_log):
    with open(arquivo_log, 'r') as log_out:
        ids = log_out.read().split()
        
    return ids

In [ ]:
def download_from_url(data, folder_path, log_file, urls_processadas):
    img_id, url = data
    save_name = str(img_id)
        
    if save_name not in urls_processadas:
        save_path = os.path.join(folder_path, save_name + ".jpg")
        
        try:
            response = requests.get(url, timeout=(5, 10))
            
            if response.status_code == 200:
                with open(save_path, 'wb') as file:
                    file.write(response.content)

                with open(log_file, 'a') as log:
                    log.write(save_name + '\n')
                    
        except Exception:
            pass

In [ ]:
download_data_users = list(zip(users_image_url['id'], users_image_url['url']))
download_data_channels = list(zip(channels_image_url['id'], channels_image_url['url']))

users_done = set(load_log(log_users))
channels_done = set(load_log(log_channels))

In [ ]:
func_users = partial(
    download_from_url,
    folder_path=users_image_path,
    log_file=log_users,
    urls_processadas=users_done
)

with ThreadPoolExecutor(max_workers=30) as executor:
    list(tqdm(executor.map(func_users, download_data_users), total=len(download_data_users)))

In [ ]:
func_channels = partial(
    download_from_url,
    folder_path=channels_image_path,
    log_file=log_channels,
    urls_processadas=channels_done
)

with ThreadPoolExecutor(max_workers=20) as executor:
    list(tqdm(executor.map(func_channels, download_data_channels), total=len(download_data_channels)))

---
## 2. Análise de Atributos Demográficos

Extrai idade e gênero das faces usando **InsightFace** (modelo buffalo_l):
- **Detecção**: det_score mínimo de 0.80 para filtrar desenhos/avatares
- **Filtro de Idade**: Remove perfis com idade < 15 anos (possíveis ruídos)
- **Validação**: Descarta imagens sem rosto ou com múltiplos rostos
- **Atributos Extraídos**:
  - **age**: Idade estimada
  - **gender**: M (masculino) ou F (feminino)
- **Output**: CSVs com ID, idade e gênero para cada usuário e canal processado

In [2]:
# criando os arquivos csv, caso ainda não exista
if not os.path.exists(csv_users_path):
    with open(csv_users_path, 'w') as f:
        f.write('id,age,gender\n')
        
    df_users_att = pd.read_csv(csv_users_path)
else:
    df_users_att = pd.read_csv(csv_users_path)

        
if not os.path.exists(csv_channels_path):
    with open(csv_channels_path, 'w') as f:
        f.write('id,age,gender\n')
        
    df_channels_att = pd.read_csv(csv_channels_path)
else:
    df_channels_att = pd.read_csv(csv_channels_path)

In [ ]:
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(224, 224))

def get_facial_attributes(path_img, path_output, set_ids):
    filename = os.path.basename(path_img)
    uid = os.path.splitext(filename)[0]

    if uid in set_ids:
        return
    
    img = cv2.imread(path_img) 
    if img is None:
        return

    faces = app.get(img)
    if len(faces) == 0 or len(faces) > 1:
        return # descarta imagens que não possuem nenhum rosto ou mais de um rosto
    
    # quando há mais de um rosto, pega-se o maior
    # f = max(faces, key=lambda face: (face.bbox[2] - face.bbox[0]) * (face.bbox[3] - face.bbox[1]))
    f = faces[0]
    
    # definindo um limite, objetivo de reduzir imagem de anime/desenho
    if f.det_score < 0.80:
        return

    # pessoas com menos de anos serão consideradas ruídos
    if f.age < 15:
        return

    gender = "M" if f.gender == 1 else "F"
    age = f.age

    with open(path_output, 'a') as o:
        o.write(f"{uid},{age},{gender}\n")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/gabrielct/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider

In [4]:
all_faces_users = [os.path.join(users_image_path, f) for f in os.listdir(users_image_path) if f.endswith('.jpg')]
ids = set(df_users_att['id'].unique())

for img in tqdm(all_faces_users):
    get_facial_attributes(img, csv_users_path, ids)

100%|██████████| 192282/192282 [40:30<00:00, 79.10it/s]  


In [5]:
all_faces_channels = [os.path.join(channels_image_path, f) for f in os.listdir(channels_image_path) if f.endswith('.jpg')]
ids = set(df_channels_att['id'].unique())

for img in tqdm(all_faces_channels):
    get_facial_attributes(img, csv_channels_path, ids)

100%|██████████| 1140/1140 [00:18<00:00, 61.28it/s]
